In [1]:
%pip install tatc
from tatc import utils
from tatc.schemas import PointedInstrument, Satellite, Instrument, TwoLineElements  

earthcare = Satellite(
    name = "EarthCare",
    orbit = TwoLineElements(
        tle=[
            "1 59908U 24101A   25200.34125573  .00010433  00000+0  14571-3 0  9999",
            "2 59908  97.0168 326.4971 0001222 108.6708 251.4681 15.57041891 64775"
        ]
    ),
    instruments=[
        #PointedInstrument(
         #   name = "CPR",
            #field_of_regard=utils.swath_width_to_field_of_regard(394e3,650),
          #  cross_track_field_of_view = utils.swath_width_to_field_of_regard(394e3, 6500),
           # along_track_field_of_view = utils.swath_width_to_field_of_regard(394e3,10000)
        #),
        PointedInstrument(
            name = "MSI",
            field_of_regard=utils.swath_width_to_field_of_regard(394e3,150e3) + 2*5.760868, #cross_track_field_of_view + 2*roll angle
            cross_track_field_of_view = utils.swath_width_to_field_of_view(394e3, 150e3, 5.760868),
            along_track_field_of_view = utils.swath_width_to_field_of_view(394e3, 10e3, 5.760868),
            roll_angle = 5.760868,  # degrees
            is_rectangular = True  
        )
    ]
)
        
satellites = [earthcare]


[notice] A new release of pip is available: 26.0.1 -> 26.1.2
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [9]:
from datetime import datetime, timezone, timedelta
from dateutil.relativedelta import relativedelta
from tatc.analysis import compute_ground_track
import pandas as pd
from joblib import Parallel, delayed

startdate = datetime(2025,7,19,15,4, tzinfo = timezone.utc)  # initial date, discard the year. 
duration = timedelta(hours = 5880)
step = timedelta(seconds = 5)
batch_duration = timedelta(minutes=10)

def compute_groundtrack(satellite, start, duration, batch_duration, time_step):
    return pd.concat(
    Parallel(-1)(
        delayed(compute_ground_track)(
            satellite,
            pd.date_range(start + i*batch_duration, start + (i+1)*batch_duration, freq=time_step, inclusive="left"),
            crs="spice"
        ) 
        for i in range( duration // batch_duration)
        for satellite in satellites
    ),
    ignore_index=True
)

ground_tracks = compute_groundtrack(satellites[0], startdate, duration, batch_duration, step)


In [ ]:
print(ground_tracks.head())
print(ground_tracks.columns)
print(len(ground_tracks))

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.animation as animation
from cartopy import crs as ccrs
from IPython.display import HTML
from matplotlib.patches import Patch

fig, ax = plt.subplots(subplot_kw={"projection": ccrs.PlateCarree()})

frame_duration = batch_duration

def animate(frame):
    ax.clear()
    time = startdate + frame*frame_duration
    tracks = ground_tracks[
        (time <= ground_tracks.time) 
        & (ground_tracks.time < time + frame_duration)
    ]
    if not tracks.empty:
        tracks.plot(ax=ax, color="r", transform=ccrs.PlateCarree())

    ax.set_global()
    ax.set_aspect("equal")
    ax.coastlines()
    ax.set_title(time)
    fig.tight_layout()

ani = animation.FuncAnimation(
    fig,
    animate, 
    frames=duration // frame_duration, 
    interval=100, 
    blit=False
)
display(HTML(ani.to_jshtml()))
plt.close()

In [18]:
grid_size = 0.5
g5nr_frame_duration = timedelta(hours=1)

In [ ]:
import rioxarray
import xarray as xr

dataset = xr.open_dataset(
    "https://opendap.nccs.nasa.gov/dods/OSSE/G5NR/Ganymed/7km/0.5000_deg/tavg/tavg01hr_2d_met3_Cx",
    decode_times=True,
)
#xr.open_dataset('https://opendap.nccs.nasa.gov/dods/OSSE/G5NR/Ganymed/7km/0.0625_deg/tavg/tavg30mn_2d_met3_Nx').to_netcdf('dataset.nc')
#dataset = xr.open_dataset('dataset.nc', decode_times=True)
dataset.rio.write_crs("epsg:4326", inplace=True)
dataset.rio.set_spatial_dims("lon", "lat", inplace=True)


In [ ]:
# decorate ground_tracks with tautot looked up from g5nr at each row's
# centroid + time. mirrors the time-mapping trick used in get_clusters so
# the lookup hits the same g5nr time index the clusters do.
def lookup_tautot(ground_tracks, dataset, startdate, g5nr_frame_duration):
    frames = ((ground_tracks.time - startdate) // g5nr_frame_duration).astype(int)
    ds_times = pd.to_datetime([
        (startdate + int(f) * g5nr_frame_duration)
            .replace(day=20, month=5, year=2006, tzinfo=None)
        for f in frames
    ])
    times_da = xr.DataArray(ds_times.values, dims="points")
    lats_da  = xr.DataArray(ground_tracks.geometry.centroid.y.values, dims="points")
    lons_da  = xr.DataArray(ground_tracks.geometry.centroid.x.values, dims="points")
    return dataset["tautot"].sel(
        time=times_da, lat=lats_da, lon=lons_da, method="nearest"
    ).values

ground_tracks["tautot"] = lookup_tautot(
    ground_tracks, dataset, startdate, g5nr_frame_duration
)
display(ground_tracks.head())

In [21]:
import numpy as np
import geopandas as gpd
from shapely.geometry import box
import scipy.ndimage as ndi

def compute_grid_cell_area(lat, lon):
    R = 6371.0  # Earth radius in km
    d2r = np.pi / 180.0
    ny, nx = len(lat), len(lon)

    dlat = abs(lat[1] - lat[0]) if len(lat) > 1 else 1.0
    dlon = abs(lon[1] - lon[0]) if len(lon) > 1 else 1.0

    lat_rad = lat * d2r
    dlat_rad = dlat * d2r
    dlon_rad = dlon * d2r

    area = np.zeros((ny, nx), dtype=np.float64)
    for i in range(ny):
        area[i, :] = (
            (R**2) * dlon_rad * 
            (np.sin(lat_rad[i] + dlat_rad / 2) - np.sin(lat_rad[i] - dlat_rad / 2))
        )
    return area


In [ ]:
import scipy.ndimage as ndi
import numpy as np
import geopandas as gpd
import pandas as pd
import xarray as xr
import time as _time
from shapely.geometry import box
from joblib import Parallel, delayed

# parallelized get_clusters: dedup frame -> g5nr-time, batched OpenDap
# fetch per chunk, then per-frame CPU work in threads.

threshold = 220  # brightness temperature threshold in Kelvin

# tunables
IO_CHUNK_TIMES = 168   # distinct g5nr hours per OpenDap request
IO_RETRIES     = 3     # exponential backoff on rate-limit / transient errors
CPU_WORKERS    = -1    # threads for per-frame work (numpy/scipy release GIL)


def _frame_ds_time(frame):
    # same time-mapping trick the original cell uses
    return (startdate + frame * g5nr_frame_duration).replace(
        day=20, month=5, year=2006, tzinfo=None
    )


def _fetch_chunk(chunk_times):
    # one batched HTTPs request via a contiguous time slice, then local pick
    t_min, t_max = min(chunk_times), max(chunk_times)
    for attempt in range(IO_RETRIES):
        try:
            sub = (
                dataset[["lwtup", "prectot"]]
                .sel(time=slice(t_min, t_max), lat=slice(-89, 89))
                .load()
            )
            break
        except Exception:
            if attempt == IO_RETRIES - 1:
                raise
            _time.sleep(2 ** attempt)
    pick = xr.DataArray(pd.to_datetime(chunk_times).values, dims="t")
    sub = sub.sel(time=pick, method="nearest")
    return {
        t: (sub["lwtup"].isel(t=j).values, sub["prectot"].isel(t=j).values)
        for j, t in enumerate(chunk_times)
    }


def _process_frame(frame, arrays, area_2d, lat2d, lon2d):
    # CPU-only: build the dissolved cluster GeoDataFrame for one frame
    try:
        lwtup_2d, prectot_2d = arrays[_frame_ds_time(frame)]

        tb = np.sqrt(np.sqrt(lwtup_2d / 5.67037e-8))
        labels, _ = ndi.label(tb < threshold)

        mask = labels > 0
        if not mask.any():
            return gpd.GeoDataFrame()

        cluster_labels = labels[mask]
        lat_vals       = lat2d[mask]
        lon_vals       = lon2d[mask]
        prectot_vals   = prectot_2d[mask]
        area_vals      = area_2d[mask]

        # single NaN filter across all columns — keeps lengths aligned
        finite = ~np.isnan(prectot_vals)
        cluster_labels = cluster_labels[finite]
        lat_vals       = lat_vals[finite]
        lon_vals       = lon_vals[finite]
        prectot_vals   = prectot_vals[finite]
        area_vals      = area_vals[finite]

        if cluster_labels.size == 0:
            return gpd.GeoDataFrame()

        cells = gpd.GeoDataFrame(
            {
                "count": 1,
                "cluster": cluster_labels,
                "lat": lat_vals,
                "lon": lon_vals,
                "time": startdate + frame * g5nr_frame_duration,
                "prectot": prectot_vals,
                "area": area_vals,
            },
            geometry=[
                box(lo, la, lo + grid_size, la + grid_size)
                for lo, la in zip(lon_vals, lat_vals)
            ],
            crs="EPSG:4326",
        )
        cells["tot_prectot"] = cells["prectot"]
        cells["avg_prectot"] = cells["prectot"]
        cells["max_prectot"] = cells["prectot"]

        return cells[cells.cluster > 0].dissolve(
            by=["time", "cluster"],
            aggfunc={
                "count": "sum",
                "area": "sum",
                "tot_prectot": "sum",
                "avg_prectot": "mean",
                "max_prectot": "max",
            },
        )
    except Exception as e:
        print(f"Frame {frame} failed: {e}")
        return gpd.GeoDataFrame()


_t0 = _time.time()
n_frames = int(duration // g5nr_frame_duration)

# precompute area + meshgrid once, shared across every frame
sample_lat = dataset["lat"].sel(lat=slice(-89, 89)).values
sample_lon = dataset["lon"].values
area_2d = compute_grid_cell_area(sample_lat, sample_lon)
lon2d, lat2d = np.meshgrid(sample_lon, sample_lat)

# dedup: many frames map to the same g5nr timestamp under the .replace mapping
unique_times = sorted({_frame_ds_time(f) for f in range(n_frames)})
print(f"{n_frames} frames -> {len(unique_times)} distinct g5nr time slices")

# group frames by which time-chunk their data lives in (bounds memory)
time_chunks = [
    unique_times[i:i + IO_CHUNK_TIMES]
    for i in range(0, len(unique_times), IO_CHUNK_TIMES)
]
time_to_chunk_idx = {t: ci for ci, c in enumerate(time_chunks) for t in c}
frames_by_chunk = [[] for _ in time_chunks]
for f in range(n_frames):
    frames_by_chunk[time_to_chunk_idx[_frame_ds_time(f)]].append(f)

# stream: fetch chunk -> process its frames in parallel -> drop arrays
all_results = []
for ci, chunk_times in enumerate(time_chunks):
    _tc = _time.time()
    arrays = _fetch_chunk(chunk_times)
    chunk_results = Parallel(n_jobs=CPU_WORKERS, backend="threading")(
        delayed(_process_frame)(f, arrays, area_2d, lat2d, lon2d)
        for f in frames_by_chunk[ci]
    )
    all_results.extend(chunk_results)
    print(f"chunk {ci+1}/{len(time_chunks)}: "
          f"{len(chunk_times)} times, {len(frames_by_chunk[ci])} frames, "
          f"{_time.time()-_tc:.1f}s")

clusters = pd.concat([r for r in all_results if not r.empty]).reset_index()
print(f"total: {_time.time()-_t0:.1f}s, {len(clusters)} cluster rows")
display(clusters)

In [ ]:
from skyfield.api import wgs84
from tatc.constants import de421, timescale

# compute solar hour based on the angle of the sun as seen at the feature centroid
clusters["solar_hour"] = clusters.apply(
    lambda r: (
        de421["earth"] + wgs84.latlon(r.geometry.centroid.y, r.geometry.centroid.x)
    )
    .at(timescale.from_datetime(r.time + frame_duration/2))
    .observe(de421["sun"])
    .apparent()
    .hadec()[0]
    .hours
    + 12,
    axis=1,
)
display(clusters)

In [ ]:
clusters["observed"] = clusters.apply(
    lambda r: 0 if ground_tracks[
        (ground_tracks.time >= r.time) 
        & (ground_tracks.time < r.time + g5nr_frame_duration)
    ].empty 
    else int(
            ground_tracks[
            (ground_tracks.time >= r.time) 
            & (ground_tracks.time < r.time + g5nr_frame_duration)
        ].dissolve().intersects(r.geometry).iloc[0]
    ),
    axis=1
)
display(clusters)
print(clusters[clusters['observed'] == 1], "features observed")

In [ ]:
import matplotlib.animation as animation
from cartopy import crs as ccrs
from IPython.display import HTML

# create a figure
fig, ax = plt.subplots(figsize=(8,4), subplot_kw={"projection": ccrs.PlateCarree()})

def animate(frame):
    ax.clear()
    time = startdate + frame*g5nr_frame_duration
    active_clusters = clusters[
        (clusters.time >= time) 
        & (clusters.time < time + g5nr_frame_duration)
    ]
    if not active_clusters.empty:
        active_clusters.plot(
            ax=ax, 
            column="observed", 
            vmin=0, 
            vmax=1, 
            cmap="RdYlGn"
        )
    track = ground_tracks[
        (ground_tracks.time >= time) 
        & (ground_tracks.time < time + g5nr_frame_duration)
    ]
    if not track.empty:
        track.dissolve().plot(ax=ax, color="black", alpha = 0.2)
    ax.set_global()
    ax.set_aspect("equal")
    ax.coastlines()
    fig.tight_layout()
    plt.legend(handles=[
        Patch(label="MSI", facecolor="black", alpha=0.8),
        Patch(label="Unobserved", facecolor="#A50026"),
        Patch(label="Observed", facecolor="#006837"),
    ], loc="lower right")

ani = animation.FuncAnimation(
    fig, 
    animate, 
    frames=duration//g5nr_frame_duration, 
    interval=100, 
    blit=False
)
display(HTML(ani.to_jshtml()))
plt.close()

In [ ]:
from skyfield.api import wgs84
from tatc.constants import de421, timescale

# compute solar hour based on the angle of the sun as seen at the feature centroid
ground_tracks["solar_hour"] = ground_tracks.apply(
    lambda r: (
        de421["earth"] + wgs84.latlon(r.geometry.centroid.y, r.geometry.centroid.x)
    )
    .at(timescale.from_datetime(r.time + frame_duration/2))
    .observe(de421["sun"])
    .apparent()
    .hadec()[0]
    .hours
    + 12,
    axis=1,
)
display(ground_tracks)

In [ ]:
# define a frame number for both datasets to aid in temporal joining
ground_tracks["frame"] = (ground_tracks.time - startdate) // g5nr_frame_duration
clusters["frame"] = (clusters.time - startdate) // g5nr_frame_duration

# perform a spatial/temporal join
joined_data = gpd.sjoin(clusters, ground_tracks, "right", "intersects", on_attribute="frame")

display(joined_data)

# aggregate results


In [ ]:
# observed-only join: each row is a 10-minute satellite pass that scooped
# at least one cluster the satellite was looking at, with prectot stats
# from those clusters and tautot stats from the track itself.
observed_clusters = clusters[clusters["observed"] == 1].copy()

joined_observed = gpd.sjoin(
    observed_clusters, ground_tracks,
    how="right",
    predicate="intersects",
    on_attribute="frame",
)

aggregated_observed = (
    joined_observed.groupby(level=0).agg({
        "geometry": "first",
        "time_left": "first",
        "solar_hour_left": "first",
        "satellite": "first",
        "count": "sum",
        "area": "sum",
        "tot_prectot": "sum",
        "avg_prectot": "mean",
        "max_prectot": "max",
        "tautot": ["sum", "mean", "max"],
    }).rename(columns={"time_left": "time", "solar_hour_left": "solar_hour"})
)

# flatten the multiindex from the tautot triple-agg
aggregated_observed.columns = [
    "_".join(c).rstrip("_") if isinstance(c, tuple) else c
    for c in aggregated_observed.columns
]
aggregated_observed = aggregated_observed.rename(columns={
    "tautot_sum":  "tot_tautot",
    "tautot_mean": "avg_tautot",
    "tautot_max":  "max_tautot",
}).set_geometry("geometry")

# drop ground-track rows that didn't intersect any observed cluster
aggregated_observed = aggregated_observed.dropna(subset=["tot_prectot"])

display(aggregated_observed)
print(len(aggregated_observed), "track rows joined with observed clusters")

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.animation as animation
import matplotlib.colors as mcolors
from cartopy import crs as ccrs
from IPython.display import HTML

fig, ax = plt.subplots(subplot_kw={"projection": ccrs.PlateCarree()})

frame_duration = batch_duration

prectot_vmin = aggregated_observed['avg_prectot'].min()
prectot_vmax = aggregated_observed['avg_prectot'].max()

# first plot to generate a colorbar
aggregated_observed.plot(
    ax=ax, 
    column="avg_prectot", 
    norm=mcolors.LogNorm(prectot_vmin, prectot_vmax),
    transform=ccrs.PlateCarree(), 
    legend=True, 
    legend_kwds={"orientation": "horizontal", "label": "Average Precipitation (kg/m$^2$/s)"}
)

def animate(frame):
    ax.clear()
    time = startdate + frame*frame_duration
    tracks = aggregated_observed[
        (aggregated_observed.time >= time) 
        & (aggregated_observed.time < time + frame_duration)
    ]
    if not tracks.empty:
        tracks.boundary.plot(ax=ax, color="r", linewidth=0.5)
        tracks.plot(
            ax=ax, 
            column="avg_prectot", 
            norm=mcolors.LogNorm(prectot_vmin, prectot_vmax),
            transform=ccrs.PlateCarree(),
        )
    ax.set_global()
    ax.set_aspect("equal")
    ax.coastlines()
    ax.set_title(time)
    fig.tight_layout()

ani = animation.FuncAnimation(
    fig,
    animate, 
    frames=duration // frame_duration, 
    interval=100, 
    blit=False
)
display(HTML(ani.to_jshtml()))
plt.close()

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

clusters_filtered = clusters[
    (clusters['time'].dt.day == 19)
]
hourly_stats = (
    clusters_filtered[:1100]
    .groupby('solar_hour')
    .agg(mean_prectot=('avg_prectot', 'mean'))
    .reset_index()
    .sort_values('solar_hour')
)

print(hourly_stats.head())
print(len(hourly_stats))

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(hourly_stats['solar_hour'], hourly_stats['mean_prectot'], marker = '.', color = 'b')
ax.set_xlabel('Solar Hour (0–23)')
ax.set_ylabel('AVG_PRECTOT (kg m⁻² s⁻¹)')
ax.set_title('Mean Precipitation vs Solar Hour for 1 month period')
ax.set_xticks(range(0, 25, 2))
plt.tight_layout()
plt.show()

#probability distribution (KDE)


In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(8, 4))

sns.kdeplot(
    data=clusters_filtered[:1100],   
    x="solar_hour",
    y="avg_prectot",
    fill=True,      
    cmap="Reds",  
    bw_adjust=0.5,       
    ax=ax,
)

ax.set_xlabel("Solar Hour (0–24)")
ax.set_ylabel("AVG_PRECTOT (kg m⁻² s⁻¹)")
ax.set_title("Mean Precipitation vs Solar Hour for 1 month period")
ax.set_xlim(0, 23)
ax.set_xticks(range(0, 25, 2))
ax.set_ylim(0, 4e-3)
plt.tight_layout()
plt.show()


In [ ]:
import seaborn as sns
import cartopy.crs as ccrs
import matplotlib.pyplot as plt
import pandas as pd

agg = aggregated_observed.copy()
agg["centroid"] = agg.geometry.centroid 

flat = pd.DataFrame({
    "lon": agg.centroid.x,
    "lat": agg.centroid.y,
    "avg_prectot": agg.avg_prectot,
    "tot_prectot": agg.tot_prectot,
    "time": agg.time,
})

fig, ax = plt.subplots(figsize=(10, 5), subplot_kw={"projection": ccrs.PlateCarree()})

sns.kdeplot(
    data=flat,
    x="lon",
    y="lat",
    weights="avg_prectot",
    fill=True,
    cmap="turbo",
    bw_adjust=0.2,
    ax=ax
)



ax.coastlines()
ax.set_xlim(-180, 180)
ax.set_ylim(-90, 90)
ax.set_title("KDE of  Simulated Mean Precipitation observable by EarthCare MSI for 8 months")
ax.set_xlabel("Longitude")
ax.set_ylabel("Latitude")
ax.set_xticks(range(-179, 179, 30))
ax.set_yticks(range(-90, 91, 30))
ax.gridlines(draw_labels=True)
#plt.tight_layout()
plt.show()

In [ ]:
import seaborn as sns
import cartopy.crs as ccrs
import matplotlib.pyplot as plt
import pandas as pd

agg = aggregated_observed.copy()
agg["centroid"] = agg.geometry.centroid 

flat = pd.DataFrame({
    "lon": agg.centroid.x,
    "lat": agg.centroid.y,
    "avg_tautot": agg.avg_tautot,
    "tot_tautot": agg.tot_tautot,
    "time": agg.time,
})

fig, ax = plt.subplots(figsize=(10, 5), subplot_kw={"projection": ccrs.PlateCarree()})

sns.kdeplot(
    data=flat,
    x="lon",
    y="lat",
    weights="avg_tautot",
    fill=True,
    cmap="turbo",
    bw_adjust=0.2,
    ax=ax
)


ax.coastlines()
ax.set_xlim(-180, 180)
ax.set_ylim(-90, 90)
ax.set_title("KDE of Average Cloud Cover Measured by EarthCare MSI for 8 months")
ax.set_xlabel("Longitude")
ax.set_ylabel("Latitude")
ax.set_xticks(range(-179, 179, 30))
ax.set_yticks(range(-90, 91, 30))
ax.gridlines(draw_labels=True)
#plt.tight_layout()
plt.show()

In [ ]:
print(f"all ground-track passes: {len(ground_tracks):>6}")
print(f"observed-only passes:    {len(aggregated_observed):>6}")
print(f"observed fraction:       {len(aggregated_observed) / len(ground_tracks):.1%}")